# The 1504-cfg analysis, 2pt part: $E_0$ for the single ratio and the gap $E_1 - E_0$, with AIC (forward)

Sections 3 and 3b of `matrix_element_analysis_1504cfg.ipynb`, moved here on 2026-11-20 (your request). They keep their section numbers. The method is `local_analysis/final_analysis/FIT_PLAN.md` section 14 ("2pt fit for E0 with AIC" and "Gap from G5G4").

- **Input:** the 2pt of every cfg that section 1 of `matrix_element_analysis_1504cfg.ipynb` saves in `{hadron}_ncfg1504/data/pt2_projected_G5G4BU_...h5`:
    - the projected 2pt of the ratio basis (GEVP t0 3, t_ref 5);
    - the diagonal G5G4 2pt with B smearing at both ends, and with U smearing at both ends.

  The two halves are averaged.
- **Fit:** $C(t) = \sum_n c_n [e^{-E_n t} + e^{-E_n (96 - t)}]$ with 2 or 3 states, every first t 2-8 and every last t up to the end of the 2pt signal.
- **AIC:** in every jackknife sample, $w \sim \exp[-(\chi^2_j + 2k + 2N_{\rm cut})/2]$; $E_0$ and the gap are averaged with these weights.
- **Output**, in `{hadron}_ncfg1504/`:
    - `pt2_aic_fits/`: every fit;
    - `results/pt2_aic_E0_gap_...h5`: $E_0$ and the gap of every sample. Section 5 of `matrix_element_analysis_1504cfg.ipynb` reads $E_0$ from this file for the single ratio.
    - `plots/3_pt2/` and `plots/3_pt2_meff_E0_gap_...pdf`.

**Order, once for each hadron:**

1. `matrix_element_analysis_1504cfg.ipynb`, sections 0 and 1 (they save the 2pt read here), and 2, 2b and 4 if you like;
2. this notebook, sections 0, 3 and 3b;
3. `matrix_element_analysis_1504cfg.ipynb`, sections 5 and 6.

Section 3 skips every (2pt, pz) whose fit file is already saved with the same settings, so it can be stopped and run again. With 13 workers it takes 7-20 min per hadron.

## 0. Settings: hadron, folders

In [ ]:
import os
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"   #one BLAS thread per process: the many small fits of section 3 run faster;  only works before numpy is imported (a fresh kernel)
os.environ["OMP_NUM_THREADS"] = "1"
import time
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.ticker import MaxNLocator, MultipleLocator
import gvar as gv
import lsqfit
from joblib import Parallel, delayed          #runs the fits of section 3 in n_workers processes at once

#hadron
hadron = "pion"              #"pion" or "proton"
project = True               #proton only: True = positive-parity files

#the data sets of matrix_element_analysis_1504cfg.ipynb:  this notebook does not read the raw files, they only give the number of cfgs in the file names;  keep them the same as there
data_sets = [{"name": "delta_stream_e", "dir": "/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta", "ncfg": 724, "ff_ncfg": 800, "cfg_shift": 0, "skip_cfg": []},
             {"name": "frontier_stream_d", "dir": "/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_frontier_streamd", "ncfg": 781, "ff_ncfg": 798, "cfg_shift": 100000, "skip_cfg": [3006]}]
ncfg = 0                     #the total number of cfgs: it goes in the names of the files
for data in data_sets:
    ncfg = ncfg + data["ncfg"] - len(data["skip_cfg"])
Lt = 96
Ls = 32

#the fits of section 3
n_workers = 13               #processes that fit at the same time (this Mac has 18 cores);  1 = one fit after the other, in this kernel
#plots
show_plots = True            #True = every figure is also drawn in the notebook;  False = only saved (PNG files and one PDF)

#directories and names:  the same folder and files as matrix_element_analysis_1504cfg.ipynb
analysis_dir = "/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/1504cfg_analysis"
if hadron == "pion":
    hadron_tag = "pion"
elif project:
    hadron_tag = "proton_projected"
else:
    hadron_tag = "proton"
save_dir = os.path.join(analysis_dir, f"{hadron}_ncfg{ncfg}")
pt2_data_file = os.path.join(save_dir, "data", f"pt2_projected_G5G4BU_forward_q0_0_0_ncfg{ncfg}_{hadron_tag}.h5")            #input: saved by section 1 of matrix_element_analysis_1504cfg.ipynb
pt2_fit_dir = os.path.join(save_dir, "pt2_aic_fits")                                                                       #section 3: one file per 2pt set and pz
pt2_result_file = os.path.join(save_dir, "results", f"pt2_aic_E0_gap_forward_q0_0_0_ncfg{ncfg}_{hadron_tag}.h5")          #section 3: E0 and the gap of every sample;  section 5 of matrix_element_analysis_1504cfg.ipynb reads E0 from it
plot_dir = os.path.join(save_dir, "plots")
plot_title = f"{hadron_tag}, forward, {ncfg} cfg"                #the start of every plot title
assert os.path.exists(pt2_data_file), f"no file {pt2_data_file}:  run sections 0 and 1 of matrix_element_analysis_1504cfg.ipynb first, with the same hadron"


#======== functions used by several cells (the same as in matrix_element_analysis_1504cfg.ipynb) ========
def jk(data, jk_axis=0):
    #jackknife mean and error
    n = data.shape[jk_axis]
    data = np.moveaxis(data, jk_axis, 0)
    mean = np.mean(data, axis=0)
    err = np.sqrt((n - 1.0) / n * np.sum((data - mean) ** 2, axis=0))
    return mean, err


def jackknife(x):
    #per-cfg data (cfg, ...) -> jackknife samples (jk, ...)
    n_cfg = x.shape[0]
    return (x.sum(axis=0) - x) / (n_cfg - 1)


def save_figure(fig, pdf, folder, name):
    #a figure of the plot cells:  saved as PNG in plot_dir/folder and as a page of the open PDF file pdf, then drawn in the notebook (show_plots) or closed
    os.makedirs(os.path.join(plot_dir, folder), exist_ok=True)
    fig.savefig(os.path.join(plot_dir, folder, f"{name}_{hadron_tag}_ncfg{ncfg}.png"), dpi=110)
    pdf.savefig(fig)
    if show_plots:
        plt.show()
    else:
        plt.close(fig)


print("hadron", hadron_tag, "|", ncfg, "cfg | 2pt of every cfg from", pt2_data_file)
print("fits in", pt2_fit_dir, "| E0 and gap saved in", pt2_result_file, "|", n_workers, "workers")

## 3. 2pt fits with AIC: $E_0$ for the single ratio and the gap $E_1 - E_0$; every fit setting is here

In [5]:
#======== 2pt fits with AIC (FIT_PLAN.md section 14):  E0 for the single ratio, and the gap E1 - E0;  every fit setting is here;  every fit is saved ========
#model:  C(t) = sum over n < n_states of c_n [exp(-E_n t) + exp(-E_n (Lt - t))],  E_n = E0 + dE_1 + ... + dE_n,  every dE > 0
#the choices of one 2pt and pz:  n_states in pt2_n_states (k = 2 n_states fit parameters);  first t in pt2_first_t;  last t from first t + k up to the end of the 2pt signal,
#  so every fit has at least k + 1 points
#the end of the 2pt signal (each 2pt and pz):  the last t up to which the relative error of C(t) is below pt2_rel_err_end at every t from t 1 on, at most pt2_max_end
#weight of a choice in jackknife sample j:  w ~ exp[-(chi2_j + 2k + 2 N_cut)/2],  chi2_j of that sample's fit (priors included),
#  N_cut = the points of t pt2_ncut_first_t ... end of the signal that its window leaves out;  normalized over the choices of this 2pt and pz
#  E0 and the gap E1 - E0 are averaged with these weights, sample by sample.  E0 of the "projected" 2pt goes into the single ratio (section 5 of matrix_element_analysis_1504cfg.ipynb);  the G5G4 2pt give the gap only
pt2_fit_sets = ["projected", "G5G4B", "G5G4U"]       #the 2pt saved by section 1 of matrix_element_analysis_1504cfg.ipynb that are fitted:  projected = ratio basis (GEVP);  G5G4B, G5G4U = diagonal G5G4, B or U smearing
pt2_n_states = [2, 3]
pt2_first_t = [2, 3, 4, 5, 6, 7, 8]
pt2_rel_err_end = 0.05
pt2_max_end = 18
pt2_ncut_first_t = 2
#priors:  E0 = the lattice dispersion arccosh[cosh m + 1 - cos p] at p = 2 pi pz / Ls with m = pt2_m_rest, width pt2_E0_width;  every dE log-normal;  every c_n
pt2_m_rest = {"pion": 0.1412, "proton": 0.4989}[hadron]
pt2_E0_width = 0.2
pt2_dE_prior = {"pion": (0.25, 0.4), "proton": (0.5, 0.5)}[hadron]     #(center, width), log-normal: the fit varies log(dE)
pt2_c_prior = (0.0, 1e4)                                                #(center, width)
pt2_svdcut = 1e-7
pt2_maxit = 20000
pt2_tol = 1e-8
pt2_fitter = "scipy_least_squares"
#single fits of the projected 2pt, drawn next to the AIC result on the E0 page of section 3b (nothing else uses them):  name -> {pz: (n_states, first t, last t)}
#  plan_single_fit = the 2pt fit of FIT_PLAN.md section 1b;  V3 = the same with 2 states and t 7-12 at pz 5-6 (the proton 2pt test of 2026-11-19)
if hadron == "pion":
    pt2_single_fits = {"plan_single_fit": {0: (3, 3, 16), 1: (3, 3, 15), 2: (3, 3, 14), 3: (3, 3, 13), 4: (3, 3, 12), 5: (3, 2, 12), 6: (3, 2, 12)}}
else:
    pt2_single_fits = {"plan_single_fit": {0: (3, 4, 14), 1: (3, 4, 14), 2: (3, 4, 14), 3: (3, 4, 14), 4: (3, 4, 12), 5: (3, 4, 10), 6: (3, 4, 10)},
                       "V3_2states_pz56_t7-12": {0: (3, 4, 14), 1: (3, 4, 14), 2: (3, 4, 14), 3: (3, 4, 14), 4: (3, 4, 12), 5: (2, 7, 12), 6: (2, 7, 12)}}
#fixed:  the fit of the jackknife mean (covariance of all cfgs) starts from the prior centers;  if it stops at pt2_maxit, it starts again from E0 and the gaps at the prior centers
#        and c_n = C(first t) exp(E0 first t) / n_states.  Every jackknife sample j is fitted with the covariance of the other cfgs (rebuilt without cfg j), starting from the fit of the mean
#a saved fit file is used again only if it was made with these settings;  to fit again, delete it (or the folder pt2_aic_fits)
pt2_settings = (f"n_states {pt2_n_states}, first t {pt2_first_t}, rel err end {pt2_rel_err_end}, max end {pt2_max_end}, N_cut from t {pt2_ncut_first_t}, m {pt2_m_rest}, "
                f"E0 width {pt2_E0_width}, dE {pt2_dE_prior}, c {pt2_c_prior}, svdcut {pt2_svdcut}, maxit {pt2_maxit}, tol {pt2_tol}, {pt2_fitter}")


def fit_2pt(y, t_points, n_states, E0_center):
    #one 2pt fit in every jackknife sample;  y (jk, n_t): the 2pt of every sample at the times t_points (integers).  Returns a dict of plain arrays
    def model(t, p):
        E = p["E0"]
        corr = 0
        for n in range(n_states):
            if n > 0:
                E = E + p["dE"][n - 1]
            corr = corr + p["c"][n] * (gv.exp(-E * t) + gv.exp(-E * (Lt - t)))
        return corr

    def prior():
        #made again after every gv.switch_gvar()
        p = gv.BufferDict()
        p["E0"] = gv.gvar(E0_center, pt2_E0_width)
        p["c"] = gv.gvar(np.full(n_states, pt2_c_prior[0]), np.full(n_states, pt2_c_prior[1]))
        p["log(dE)"] = gv.log(gv.gvar(np.full(n_states - 1, float(pt2_dE_prior[0])), np.full(n_states - 1, float(pt2_dE_prior[1]))))
        return p

    n_jk = y.shape[0]
    y_mean = y.mean(axis=0)
    cov = (n_jk - 1) * np.cov(y, rowvar=False, bias=True)          #jackknife covariance of the mean, all cfgs
    #the fit of the jackknife mean
    gv.switch_gvar()
    fit = lsqfit.nonlinear_fit(data=(t_points, gv.gvar(y_mean, cov)), fcn=model, prior=prior(), svdcut=pt2_svdcut, maxit=pt2_maxit, tol=pt2_tol, fitter=pt2_fitter)
    restarted = fit.stopping_criterion == 0
    if restarted:
        p0 = {"E0": E0_center, "c": np.full(n_states, y_mean[0] * np.exp(E0_center * t_points[0]) / n_states), "log(dE)": np.log(np.full(n_states - 1, float(pt2_dE_prior[0])))}
        fit = lsqfit.nonlinear_fit(data=(t_points, gv.gvar(y_mean, cov)), fcn=model, prior=prior(), p0=p0, svdcut=pt2_svdcut, maxit=pt2_maxit, tol=pt2_tol, fitter=pt2_fitter)
    n_not_converged = int(fit.stopping_criterion == 0)
    chi2_mean = float(fit.chi2)
    dof = int(fit.dof)
    p0 = fit.pmean
    gv.restore_gvar()
    #every jackknife sample, with the covariance of the other cfgs:  the per-cfg values are made back from the samples,
    #  and cfg j is taken out of their sum of squares (the same as np.cov of the other cfgs, / (number of cfgs - 1))
    y_cfg = n_jk * y_mean - (n_jk - 1) * y                         #(cfg, n_t)
    y_c = y_cfg - y_cfg.mean(axis=0)
    Q = y_c.T @ y_c
    m = n_jk - 1
    E_jk = np.full((n_jk, 3), np.nan)       #(jk, state): E0, E1, E2;  NaN for a state the fit does not have
    c_jk = np.full((n_jk, 3), np.nan)       #(jk, state)
    chi2_jk = np.zeros(n_jk)
    for j in range(n_jk):
        cov_j = (Q - (1 + 1 / m) * np.outer(y_c[j], y_c[j])) / (m - 1) / (n_jk - 1)
        gv.switch_gvar()
        fit_j = lsqfit.nonlinear_fit(data=(t_points, gv.gvar(y[j], cov_j)), fcn=model, prior=prior(), p0=p0, svdcut=pt2_svdcut, maxit=pt2_maxit, tol=pt2_tol, fitter=pt2_fitter)
        n_not_converged = n_not_converged + int(fit_j.stopping_criterion == 0)
        E = fit_j.pmean["E0"]
        for n in range(n_states):
            if n > 0:
                E = E + fit_j.pmean["dE"][n - 1]
            E_jk[j, n] = E
            c_jk[j, n] = fit_j.pmean["c"][n]
        chi2_jk[j] = fit_j.chi2
        gv.restore_gvar()
    return {"E_jk": E_jk, "c_jk": c_jk, "chi2_jk": chi2_jk, "chi2_mean": chi2_mean, "dof": dof, "n_bad": n_not_converged, "restarted": restarted}


#---- the 2pt saved by section 1 of matrix_element_analysis_1504cfg.ipynb, and the E0 prior centers
time_start = time.time()
with h5py.File(pt2_data_file, "r") as f:
    pt2_saved = {}
    for name in pt2_fit_sets:
        pt2_saved[name] = f[f"C2_{name}_fb_cfg"][()]          #(cfg, pz, t)
    pz_saved = f["pz_list"][()]
n_pz = len(pz_saved)
p_lat = 2 * np.pi * pz_saved / Ls
E0_center = np.arccosh(np.cosh(pt2_m_rest) + 1 - np.cos(p_lat))     #(pz,): the center of the E0 prior
os.makedirs(pt2_fit_dir, exist_ok=True)

#---- every choice of every 2pt and pz;  a (2pt, pz) whose file is saved with these settings is not fitted again
for name in pt2_fit_sets:
    data_jk = jackknife(pt2_saved[name])                            #(jk, pz, t)
    for ipz, pz in enumerate(pz_saved):
        m, e = jk(data_jk[:, ipz])
        end = 1
        while end + 1 <= pt2_max_end and e[end + 1] / abs(m[end + 1]) < pt2_rel_err_end:
            end = end + 1
        choices = []                                                #(n_states, first t, last t)
        for n_states in pt2_n_states:
            for first in pt2_first_t:
                for last in range(first + 2 * n_states, end + 1):
                    choices.append((n_states, first, last))
        out_file = os.path.join(pt2_fit_dir, f"pt2_aic_{name}_pz{pz}_ncfg{ncfg}_{hadron_tag}.npz")
        if os.path.exists(out_file):
            saved = np.load(out_file)
            assert str(saved["settings"]) == pt2_settings and np.array_equal(saved["choices"], choices), f"{out_file} was made with other settings: delete it to fit again"
            print(f"{name} pz {pz}: already fitted, {len(choices)} choices")
            continue
        time_pz = time.time()
        results = Parallel(n_jobs=n_workers)(delayed(fit_2pt)(data_jk[:, ipz][:, np.arange(first, last + 1)], np.arange(first, last + 1), n_states, E0_center[ipz])
                                             for n_states, first, last in choices)
        np.savez(out_file, choices=np.array(choices), signal_end=end,
                 E_jk=np.stack([r["E_jk"] for r in results], axis=1),               #(jk, choice, state)
                 c_jk=np.stack([r["c_jk"] for r in results], axis=1),               #(jk, choice, state)
                 chi2_jk=np.stack([r["chi2_jk"] for r in results], axis=1),         #(jk, choice)
                 chi2_mean=np.array([r["chi2_mean"] for r in results]), dof=np.array([r["dof"] for r in results]),
                 n_bad=np.array([r["n_bad"] for r in results]), restarted=np.array([r["restarted"] for r in results]), settings=pt2_settings)
        print(f"{name} pz {pz}: {len(choices)} choices, end of the signal t {end}, {sum(r['n_bad'] for r in results)} fits stopped at maxit, "
              f"{sum(r['restarted'] for r in results)} fits of the mean started again;  {time.time() - time_pz:.0f} s", flush=True)

#---- the single fits of the projected 2pt
data_jk = jackknife(pt2_saved["projected"])
for label in pt2_single_fits:
    out_file = os.path.join(pt2_fit_dir, f"pt2_single_{label}_projected_ncfg{ncfg}_{hadron_tag}.npz")
    choices = [pt2_single_fits[label][int(pz)] for pz in pz_saved]
    if os.path.exists(out_file):
        saved = np.load(out_file)
        assert str(saved["settings"]) == pt2_settings and np.array_equal(saved["choices"], choices), f"{out_file} was made with other settings: delete it to fit again"
        print(f"single fit {label}: already fitted")
        continue
    results = Parallel(n_jobs=n_workers)(delayed(fit_2pt)(data_jk[:, ipz][:, np.arange(first, last + 1)], np.arange(first, last + 1), n_states, E0_center[ipz])
                                         for ipz, (n_states, first, last) in enumerate(choices))
    np.savez(out_file, choices=np.array(choices), E_jk=np.stack([r["E_jk"] for r in results], axis=1), c_jk=np.stack([r["c_jk"] for r in results], axis=1),
             chi2_mean=np.array([r["chi2_mean"] for r in results]), dof=np.array([r["dof"] for r in results]), n_bad=np.array([r["n_bad"] for r in results]),
             restarted=np.array([r["restarted"] for r in results]), settings=pt2_settings)
    print(f"single fit {label}: {sum(r['n_bad'] for r in results)} fits stopped at maxit, chi2/dof of the mean", " ".join(f"{r['chi2_mean'] / r['dof']:.2f}" for r in results))

#---- the AIC average of every 2pt, in every jackknife sample
pt2_E0_jk = {}          #pt2_E0_jk[name]: (jk, pz), E0 averaged with the AIC weights of each sample
pt2_gap_jk = {}         #pt2_gap_jk[name]: (jk, pz), E1 - E0 averaged the same way
pt2_weight_mean = {}    #pt2_weight_mean[name][pz]: (choice,), the weights, mean over the samples
for name in pt2_fit_sets:
    pt2_E0_jk[name] = np.zeros((data_jk.shape[0], n_pz))
    pt2_gap_jk[name] = np.zeros((data_jk.shape[0], n_pz))
    pt2_weight_mean[name] = {}
    for ipz, pz in enumerate(pz_saved):
        d = np.load(os.path.join(pt2_fit_dir, f"pt2_aic_{name}_pz{pz}_ncfg{ncfg}_{hadron_tag}.npz"))
        choices = d["choices"]
        end = int(d["signal_end"])
        k = 2 * choices[:, 0]
        n_cut = (end - pt2_ncut_first_t + 1) - (choices[:, 2] - choices[:, 1] + 1)
        aic = d["chi2_jk"] + 2 * k + 2 * n_cut                                    #(jk, choice)
        weight = np.exp(-0.5 * (aic - aic.min(axis=1, keepdims=True)))
        weight = weight / weight.sum(axis=1, keepdims=True)
        pt2_E0_jk[name][:, ipz] = (weight * d["E_jk"][:, :, 0]).sum(axis=1)
        pt2_gap_jk[name][:, ipz] = (weight * (d["E_jk"][:, :, 1] - d["E_jk"][:, :, 0])).sum(axis=1)
        pt2_weight_mean[name][int(pz)] = weight.mean(axis=0)
pt2_single_E0_jk = {}   #pt2_single_E0_jk[label]: (jk, pz)
for label in pt2_single_fits:
    pt2_single_E0_jk[label] = np.load(os.path.join(pt2_fit_dir, f"pt2_single_{label}_projected_ncfg{ncfg}_{hadron_tag}.npz"))["E_jk"][:, :, 0]

#---- save
os.makedirs(os.path.dirname(pt2_result_file), exist_ok=True)
if os.path.exists(pt2_result_file):
    print("replacing", pt2_result_file)
with h5py.File(pt2_result_file, "w") as f:
    for name in pt2_fit_sets:
        f[f"{name}/E0_jk"] = pt2_E0_jk[name]            #(jk, pz):  projected/E0_jk is the E0 of the single ratio
        f[f"{name}/gap_jk"] = pt2_gap_jk[name]          #(jk, pz):  E1 - E0
        for pz in pt2_weight_mean[name]:
            f[f"{name}/weight_mean_pz{pz}"] = pt2_weight_mean[name][pz]
            f[f"{name}/choices_pz{pz}"] = np.load(os.path.join(pt2_fit_dir, f"pt2_aic_{name}_pz{pz}_ncfg{ncfg}_{hadron_tag}.npz"))["choices"]
    for label in pt2_single_fits:
        f[f"single/{label}/E0_jk"] = pt2_single_E0_jk[label]
        f[f"single/{label}/choices"] = [pt2_single_fits[label][int(pz)] for pz in pz_saved]
    f["settings"] = pt2_settings
    f["E0_prior_center"] = E0_center
    f["pz_list"] = pz_saved
    f["ncfg"] = ncfg
    f["hadron_tag"] = hadron_tag
print("saved", pt2_result_file)

#---- print:  E0 against the dispersion relations, with m = E0 at pz 0 of the same sample;  the gaps
m_jk = pt2_E0_jk["projected"][:, :1]                                        #(jk, 1)
lattice_jk = np.arccosh(np.cosh(m_jk) + 1 - np.cos(p_lat))                  #(jk, pz)
continuum_jk = np.sqrt(m_jk ** 2 + p_lat ** 2)
print("E0 of the projected 2pt (AIC);  difference from the lattice dispersion and from sqrt(m^2 + p^2), m = E0(pz 0) of the same sample, in units of its error")
for ipz, pz in enumerate(pz_saved):
    E_m, E_e = jk(pt2_E0_jk["projected"][:, ipz])
    text = f"pz {pz}: E0 {str(gv.gvar(E_m, E_e)):>12s}"
    if pz > 0:
        d_m, d_e = jk(pt2_E0_jk["projected"][:, ipz] - lattice_jk[:, ipz])
        c_m, c_e = jk(pt2_E0_jk["projected"][:, ipz] - continuum_jk[:, ipz])
        text = text + f",  lattice {d_m / d_e:+5.1f},  continuum {c_m / c_e:+5.1f}"
    for label in pt2_single_fits:
        s_m, s_e = jk(pt2_single_E0_jk[label][:, ipz])
        text = text + f";  {label} {gv.gvar(s_m, s_e)}"
    print(text)
print("the gap E1 - E0 (AIC), pz 0-6:")
for name in pt2_fit_sets:
    g_m, g_e = jk(pt2_gap_jk[name])
    print(f"{name:>10s}:", " ".join(f"{str(gv.gvar(g_m[i], g_e[i])):>11s}" for i in range(n_pz)))
print(f"{time.time() - time_start:.0f} s")

projected pz 0: 126 choices, end of the signal t 18, 0 fits stopped at maxit, 1 fits of the mean started again;  30 s
projected pz 1: 126 choices, end of the signal t 18, 0 fits stopped at maxit, 6 fits of the mean started again;  25 s
projected pz 2: 126 choices, end of the signal t 18, 0 fits stopped at maxit, 8 fits of the mean started again;  29 s
projected pz 3: 126 choices, end of the signal t 18, 0 fits stopped at maxit, 0 fits of the mean started again;  38 s
projected pz 4: 98 choices, end of the signal t 16, 0 fits stopped at maxit, 9 fits of the mean started again;  26 s
projected pz 5: 56 choices, end of the signal t 13, 0 fits stopped at maxit, 0 fits of the mean started again;  13 s
projected pz 6: 43 choices, end of the signal t 12, 0 fits stopped at maxit, 0 fits of the mean started again;  10 s


/opt/anaconda3/lib/python3.14/site-packages/scipy/optimize/_lsq/trf.py:527: RuntimeWarning: overflow encountered in dot
  cost_new = 0.5 * np.dot(f_new, f_new)
/opt/anaconda3/lib/python3.14/site-packages/lsqfit/__init__.py:715: UserWarning: det(fit.cov) < 0 --- roundoff errors? Try an svd cut.
  warnings.warn('det(fit.cov) < 0 --- roundoff errors? Try an svd cut.')
/var/folders/fd/b0prbcvx72l2qq4ncklr4nq40000gn/T/ipykernel_17737/2206912482.py:46: RuntimeWarning: overflow encountered in exp
/var/folders/fd/b0prbcvx72l2qq4ncklr4nq40000gn/T/ipykernel_17737/2206912482.py:46: RuntimeWarning: invalid value encountered in add
/opt/anaconda3/lib/python3.14/site-packages/lsqfit/__init__.py:715: UserWarning: det(fit.cov) < 0 --- roundoff errors? Try an svd cut.
  warnings.warn('det(fit.cov) < 0 --- roundoff errors? Try an svd cut.')


G5G4B pz 0: 126 choices, end of the signal t 18, 0 fits stopped at maxit, 5 fits of the mean started again;  31 s


/var/folders/fd/b0prbcvx72l2qq4ncklr4nq40000gn/T/ipykernel_17737/2206912482.py:46: RuntimeWarning: overflow encountered in exp
/var/folders/fd/b0prbcvx72l2qq4ncklr4nq40000gn/T/ipykernel_17737/2206912482.py:46: RuntimeWarning: invalid value encountered in add
/opt/anaconda3/lib/python3.14/site-packages/lsqfit/__init__.py:715: UserWarning: det(fit.cov) < 0 --- roundoff errors? Try an svd cut.
  warnings.warn('det(fit.cov) < 0 --- roundoff errors? Try an svd cut.')
/opt/anaconda3/lib/python3.14/site-packages/scipy/optimize/_lsq/trf.py:527: RuntimeWarning: overflow encountered in dot
  cost_new = 0.5 * np.dot(f_new, f_new)
/opt/anaconda3/lib/python3.14/site-packages/lsqfit/__init__.py:715: UserWarning: det(fit.cov) < 0 --- roundoff errors? Try an svd cut.
  warnings.warn('det(fit.cov) < 0 --- roundoff errors? Try an svd cut.')


G5G4B pz 1: 126 choices, end of the signal t 18, 0 fits stopped at maxit, 6 fits of the mean started again;  26 s


/var/folders/fd/b0prbcvx72l2qq4ncklr4nq40000gn/T/ipykernel_17737/2206912482.py:46: RuntimeWarning: overflow encountered in exp
/var/folders/fd/b0prbcvx72l2qq4ncklr4nq40000gn/T/ipykernel_17737/2206912482.py:46: RuntimeWarning: invalid value encountered in add
/opt/anaconda3/lib/python3.14/site-packages/scipy/optimize/_lsq/trf.py:527: RuntimeWarning: overflow encountered in dot
  cost_new = 0.5 * np.dot(f_new, f_new)


G5G4B pz 2: 126 choices, end of the signal t 18, 0 fits stopped at maxit, 2 fits of the mean started again;  27 s
G5G4B pz 3: 126 choices, end of the signal t 18, 0 fits stopped at maxit, 0 fits of the mean started again;  27 s
G5G4B pz 4: 84 choices, end of the signal t 15, 0 fits stopped at maxit, 0 fits of the mean started again;  17 s
G5G4B pz 5: 56 choices, end of the signal t 13, 0 fits stopped at maxit, 0 fits of the mean started again;  14 s
G5G4B pz 6: 43 choices, end of the signal t 12, 0 fits stopped at maxit, 0 fits of the mean started again;  10 s


/opt/anaconda3/lib/python3.14/site-packages/scipy/optimize/_lsq/trf.py:527: RuntimeWarning: overflow encountered in dot
  cost_new = 0.5 * np.dot(f_new, f_new)


G5G4U pz 0: 126 choices, end of the signal t 18, 0 fits stopped at maxit, 8 fits of the mean started again;  30 s


/opt/anaconda3/lib/python3.14/site-packages/scipy/optimize/_lsq/trf.py:527: RuntimeWarning: overflow encountered in dot
  cost_new = 0.5 * np.dot(f_new, f_new)
/var/folders/fd/b0prbcvx72l2qq4ncklr4nq40000gn/T/ipykernel_17737/2206912482.py:46: RuntimeWarning: overflow encountered in exp
/var/folders/fd/b0prbcvx72l2qq4ncklr4nq40000gn/T/ipykernel_17737/2206912482.py:46: RuntimeWarning: invalid value encountered in add
/opt/anaconda3/lib/python3.14/site-packages/scipy/optimize/_lsq/trf.py:527: RuntimeWarning: overflow encountered in dot
  cost_new = 0.5 * np.dot(f_new, f_new)


KeyboardInterrupt: 

## 3b. Plots of the 2pt: effective mass with $E_0$, $E_0$ against the dispersion relations, AIC weights, gap against the fit range

In [ ]:
#======== plots of the 2pt:  effective mass with the fitted E0;  E0 against the dispersion relations;  the AIC weights;  the gap against the fit range;  saved as PNG and in one PDF ========
#reads pt2_data_file (section 1 of matrix_element_analysis_1504cfg.ipynb) and the files of section 3;  after a restart run sections 0 and 3 of this notebook first (section 3 then fits nothing again: every fit is read from its file)
#effective mass:  m_eff(t) = log[C(t)/C(t+1)] of the projected 2pt, every t (C is stored to t 19)
#  solid band: the fitted m_eff of the 2pt AIC, made in every jackknife sample (the AIC-weighted curves of all choices), then mean and jackknife error at each t,
#              from the first t of the largest-weight choice to the end of the 2pt signal;  hatched band: E0 with its jackknife error
#  dash-dot line: lattice dispersion arccosh[cosh m + 1 - cos p], dashed: sqrt(m^2 + p^2), both with m = E0 at pz 0 (mean over the samples)
pt2_plot_ylim = {"pion": (0.0, 1.95), "proton": (0.0, 2.15)}[hadron]
pt2_plot_t_max = 20
pz_colors = ["#52514e", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6"]
pz_markers = ["o", "s", "^", "D", "v", "P", "X"]
first_t_colors = {2: "#2a78d6", 3: "#eb6834", 4: "#1baf7a", 5: "#eda100", 6: "#e87ba4", 7: "#008300", 8: "#6250d6"}
pt2_set_text = {"projected": "the projected 2pt (ratio basis, GEVP t0 3, t_ref 5)",
                "G5G4B": "the G5G4 2pt (B smearing at both ends, no GEVP)",
                "G5G4U": "the G5G4 2pt (U smearing at both ends, no GEVP)"}

with h5py.File(pt2_data_file, "r") as f:
    C2_proj = f["C2_projected_fb_cfg"][()]              #(cfg, pz, t)
    pz_saved = f["pz_list"][()]
n_pz = len(pz_saved)
p_lat = 2 * np.pi * pz_saved / Ls
with h5py.File(pt2_result_file, "r") as f:
    E0_aic_jk = f["projected/E0_jk"][()]                #(jk, pz)
    single_E0_jk = {}
    for label in pt2_single_fits:
        single_E0_jk[label] = f[f"single/{label}/E0_jk"][()]
pt2_fits = {}           #pt2_fits[name][pz]: the saved fits of every choice (section 3), and their weights (jk, choice)
for name in pt2_fit_sets:
    pt2_fits[name] = {}
    for pz in pz_saved:
        d = dict(np.load(os.path.join(pt2_fit_dir, f"pt2_aic_{name}_pz{pz}_ncfg{ncfg}_{hadron_tag}.npz")))
        n_cut = (int(d["signal_end"]) - pt2_ncut_first_t + 1) - (d["choices"][:, 2] - d["choices"][:, 1] + 1)
        aic = d["chi2_jk"] + 2 * (2 * d["choices"][:, 0]) + 2 * n_cut
        weight = np.exp(-0.5 * (aic - aic.min(axis=1, keepdims=True)))
        d["weight"] = weight / weight.sum(axis=1, keepdims=True)         #(jk, choice), as in section 3
        pt2_fits[name][int(pz)] = d
os.makedirs(plot_dir, exist_ok=True)
pdf_file = os.path.join(plot_dir, f"3_pt2_meff_E0_gap_{hadron_tag}_ncfg{ncfg}.pdf")
with PdfPages(pdf_file) as pdf:
    #---- page 1: the effective mass of every pz
    C2_jk = jackknife(C2_proj)                                          #(jk, pz, t)
    with np.errstate(invalid="ignore", divide="ignore"):
        meff_2pt = np.log(C2_jk[:, :, :-1] / C2_jk[:, :, 1:])          #(jk, pz, t): t = 0 ... 18
    meff_2pt_mean, meff_2pt_err = jk(meff_2pt)
    E0_mean, E0_err = jk(E0_aic_jk)
    m_rest = E0_aic_jk[:, 0]                                            #(jk,): the rest mass of every sample = E0 at pz 0
    lattice_jk = np.arccosh(np.cosh(m_rest[:, None]) + 1 - np.cos(p_lat))      #(jk, pz)
    continuum_jk = np.sqrt(m_rest[:, None] ** 2 + p_lat ** 2)
    fig, ax = plt.subplots(figsize=(11.69, 8.27))
    t_all = np.arange(meff_2pt.shape[2])
    for ipz, pz in enumerate(pz_saved):
        ok = np.isfinite(meff_2pt_mean[ipz]) & (t_all >= 1)
        ax.errorbar(t_all[ok] + 0.04 * (ipz - 3), meff_2pt_mean[ipz, ok], meff_2pt_err[ipz, ok], color=pz_colors[ipz], marker=pz_markers[ipz], ms=4, lw=1.2, elinewidth=0.9, capsize=2,
                    label=f"pz {pz}: E0 (AIC) {gv.gvar(E0_mean[ipz], E0_err[ipz])}")
        #the fitted m_eff of every sample: the AIC-weighted average over the choices of log[C_fit(t)/C_fit(t+1)]
        d = pt2_fits["projected"][int(pz)]
        E_all = np.nan_to_num(d["E_jk"])                                #(jk, choice, state);  a 2-state fit has c = 0 for the third state
        c_all = np.nan_to_num(d["c_jk"])
        best = int(np.argmax(d["weight"].mean(axis=0)))
        t_band = np.linspace(d["choices"][best][1], int(d["signal_end"]), 10 * (int(d["signal_end"]) - d["choices"][best][1]) + 1)
        band = np.zeros((E_all.shape[0], len(t_band)))                  #(jk, t)
        for i, t in enumerate(t_band):
            C_t = (c_all * (np.exp(-E_all * t) + np.exp(-E_all * (Lt - t)))).sum(axis=2)                 #(jk, choice)
            C_t1 = (c_all * (np.exp(-E_all * (t + 1)) + np.exp(-E_all * (Lt - t - 1)))).sum(axis=2)
            with np.errstate(invalid="ignore", divide="ignore"):
                band[:, i] = (d["weight"] * np.log(C_t / C_t1)).sum(axis=1)
        band_mean, band_err = jk(band)
        ax.fill_between(t_band, band_mean - band_err, band_mean + band_err, color=pz_colors[ipz], alpha=0.30, lw=0)
        ax.plot(t_band, band_mean, color=pz_colors[ipz], lw=1.0)
        ax.fill_between([t_band[0], pt2_plot_t_max], E0_mean[ipz] - E0_err[ipz], E0_mean[ipz] + E0_err[ipz], facecolor="none", edgecolor=pz_colors[ipz], hatch="////", lw=0.6, alpha=0.8)
        ax.axhline(lattice_jk[:, ipz].mean(), color=pz_colors[ipz], ls="-.", lw=0.8, alpha=0.7)
        ax.axhline(continuum_jk[:, ipz].mean(), color=pz_colors[ipz], ls="--", lw=0.8, alpha=0.7)
    ax.grid(color="#e1e0d9", lw=0.5)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=8)
    ax.xaxis.set_major_locator(MultipleLocator(1))
    ax.set_xlim(0, pt2_plot_t_max)
    ax.set_ylim(*pt2_plot_ylim)
    ax.set_xlabel("t", fontsize=10)
    ax.set_ylabel(r"$m_{\rm eff}(t) = \log[C(t)/C(t+1)]$", fontsize=10)
    ax.legend(fontsize=8, frameon=False, ncol=2, loc="upper right", title="E0: mean and jackknife error of the per-sample AIC average", title_fontsize=8)
    fig.suptitle(f"{plot_title}: 2pt effective mass log[C(t)/C(t+1)] of {pt2_set_text['projected']}, every pz, every t\n"
                 "solid bands: fitted m_eff of the 2pt AIC, made in every jackknife sample, then mean and jackknife error at each t\n"
                 "(from the first t of the largest-weight window to the end of the signal);  hatched: E0 with its jackknife error\n"
                 f"dash-dot: lattice dispersion, dashed: $\\sqrt{{m^2 + p^2}}$, both with m = E0(pz 0) = {gv.gvar(E0_mean[0], E0_err[0])}", fontsize=9.5)
    fig.tight_layout(rect=(0, 0, 1, 0.89))
    save_figure(fig, pdf, "3_pt2", "pt2_meff")

    #---- page 2: E0 minus the dispersion relations, m = E0 at pz 0 of the same fit in every sample (pz 0 is 0 by construction)
    fig, axes = plt.subplots(1, 2, figsize=(11.69, 8.27))
    rows = [("2pt AIC", E0_aic_jk, "#2a78d6", "o")]
    row_colors = ["#52514e", "#eb6834", "#1baf7a"]
    row_markers = ["s", "^", "D"]
    for i, label in enumerate(single_E0_jk):
        rows.append((label, single_E0_jk[label], row_colors[i], row_markers[i]))
    for c, ax in enumerate(axes):
        for i, (label, E_jk_row, color, marker) in enumerate(rows):
            m_row = E_jk_row[:, 0]                                      #its own m = E0(pz 0), sample by sample
            if c == 0:
                ref_jk = np.arccosh(np.cosh(m_row[:, None]) + 1 - np.cos(p_lat))
            else:
                ref_jk = np.sqrt(m_row[:, None] ** 2 + p_lat ** 2)
            diff_mean, diff_err = jk(E_jk_row - ref_jk)
            ax.errorbar(np.arange(n_pz) + 0.1 * (i - 1), diff_mean, diff_err, color=color, marker=marker, ms=5, ls="none", capsize=2, label=label)
            for ipz in range(1, n_pz):
                ax.annotate(f"{diff_mean[ipz] / diff_err[ipz]:+.1f}", (ipz + 0.1 * (i - 1), diff_mean[ipz]), xytext=(4, -10 - 9 * i), textcoords="offset points", fontsize=6.5, color=color)
        if c == 0:
            ax.plot(np.arange(n_pz), continuum_jk.mean(axis=0) - lattice_jk.mean(axis=0), color="#898781", ls="--", lw=1, label="the other dispersion relation (2pt AIC m)")
            ax.set_title("E0 minus the lattice dispersion arccosh[cosh m + 1 - cos p];  numbers: difference / error", fontsize=9)
        else:
            ax.plot(np.arange(n_pz), lattice_jk.mean(axis=0) - continuum_jk.mean(axis=0), color="#898781", ls="--", lw=1, label="the other dispersion relation (2pt AIC m)")
            ax.set_title(r"E0 minus the continuum $\sqrt{m^2 + p^2}$;  numbers: difference / error", fontsize=9)
        ax.axhline(0, color="#898781", lw=0.6)
        ax.grid(color="#e1e0d9", lw=0.5)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
        ax.tick_params(labelsize=8)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.set_xlabel("pz", fontsize=9)
        ax.set_ylabel("E0 - dispersion", fontsize=9)
    axes[0].legend(fontsize=8, frameon=False, loc="lower left")
    fig.suptitle(f"{plot_title}: E0 of the 2pt fits against the dispersion relations;  m = E0 at pz 0 of the same fit in every jackknife sample\n"
                 f"(2pt AIC: m = {gv.gvar(E0_mean[0], E0_err[0])}), so pz 0 is 0 by construction;  errors: jackknife of the difference", fontsize=10)
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    save_figure(fig, pdf, "3_pt2", "pt2_E0_dispersion")

    #---- page 3: the AIC weights of the projected 2pt, every (first t, last t), 2 and 3 states;  x: the largest weight
    fig, axes = plt.subplots(len(pt2_n_states), n_pz, figsize=(11.69, 8.27), squeeze=False)
    for ipz, pz in enumerate(pz_saved):
        d = pt2_fits["projected"][int(pz)]
        choices = d["choices"]
        weight = d["weight"].mean(axis=0)
        E0_choice = jk(d["E_jk"][:, :, 0])[0]
        for r, n_states in enumerate(pt2_n_states):
            ax = axes[r, ipz]
            image = np.full((len(pt2_first_t), pt2_max_end + 1), np.nan)            #(first t, last t)
            for (n_s, first, last), w in zip(choices, weight):
                if n_s == n_states:
                    image[first - pt2_first_t[0], last] = w
            ax.imshow(image, cmap="Blues", vmin=0, vmax=max(weight.max(), 1e-3), aspect="auto", origin="lower",
                      extent=(-0.5, pt2_max_end + 0.5, pt2_first_t[0] - 0.5, pt2_first_t[-1] + 0.5))
            i_best = int(np.argmax(weight))
            if choices[i_best][0] == n_states:
                ax.plot(choices[i_best][2], choices[i_best][1], "x", color="#e34948", ms=6)
            ax.set_xlim(5.5, pt2_max_end + 0.5)
            ax.tick_params(labelsize=6)
            ax.xaxis.set_major_locator(MaxNLocator(integer=True))
            ax.yaxis.set_major_locator(MaxNLocator(integer=True))
            ax.set_title(f"pz {pz}, {n_states} states: weight {weight[choices[:, 0] == n_states].sum():.2f}", fontsize=7)
            if r == len(pt2_n_states) - 1:
                ax.set_xlabel("last t", fontsize=7)
            if ipz == 0:
                ax.set_ylabel("first t", fontsize=7)
        i_best = int(np.argmax(weight))
        axes[0, ipz].text(0.02, 0.98, f"largest: {choices[i_best][0]} st {choices[i_best][1]}-{choices[i_best][2]}\n({weight[i_best]:.2f}), E0 {E0_choice[i_best]:.4f}",
                          transform=axes[0, ipz].transAxes, fontsize=6, va="top")
    fig.suptitle(f"{plot_title}: weights of the 2pt AIC of {pt2_set_text['projected']} (mean over samples), every (first t, last t);  x: the largest weight", fontsize=10)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    save_figure(fig, pdf, "3_pt2", "pt2_aic_weights")

    #---- pages 4-6: the gap E1 - E0 of every choice against the fit range, one row per pz;  left 2 states, right 3 states;  dot size = AIC weight;  grey band = the AIC average
    for name in pt2_fit_sets:
        fig, axes = plt.subplots(n_pz, len(pt2_n_states), figsize=(11.69, 16.5), sharex=True, squeeze=False)
        for ipz, pz in enumerate(pz_saved):
            d = pt2_fits[name][int(pz)]
            choices = d["choices"]
            gap = d["E_jk"][:, :, 1] - d["E_jk"][:, :, 0]                   #(jk, choice)
            gap_mean, gap_err = jk(gap)
            avg_mean, avg_err = jk((d["weight"] * gap).sum(axis=1))         #the AIC average of the gap, sample by sample
            weight = d["weight"].mean(axis=0)
            ok = gap_err < 0.5
            if ok.sum() > 3:
                low, high = np.percentile(gap_mean[ok], [2, 98])
            else:
                low, high = gap_mean.min(), gap_mean.max()
            low = min(low, avg_mean - avg_err)
            high = max(high, avg_mean + avg_err)
            for c, n_states in enumerate(pt2_n_states):
                ax = axes[ipz, c]
                ax.axhspan(avg_mean - avg_err, avg_mean + avg_err, color="#898781", alpha=0.25, lw=0, label=f"AIC average of all fits: {gv.gvar(avg_mean, avg_err)}")
                for first in pt2_first_t:
                    sel = (choices[:, 0] == n_states) & (choices[:, 1] == first)
                    if not sel.any():
                        continue
                    x = choices[sel, 2] + 0.07 * (first - 5)
                    ax.errorbar(x, gap_mean[sel], gap_err[sel], color=first_t_colors[first], ls="none", elinewidth=0.7, capsize=1, zorder=2)
                    ax.scatter(x, gap_mean[sel], s=6 + 90 * weight[sel] / weight.max(), color=first_t_colors[first], edgecolors="#52514e", linewidths=0.3, zorder=3, label=f"first t {first}")
                pad = 0.15 * (high - low) + 0.01
                ax.set_ylim(low - pad, high + pad)
                ax.grid(color="#e1e0d9", lw=0.5)
                ax.set_axisbelow(True)
                ax.spines[["top", "right"]].set_visible(False)
                ax.tick_params(labelsize=6.5)
                ax.xaxis.set_major_locator(MaxNLocator(integer=True))
                ax.set_title(f"pz {pz}, {n_states} states (AIC weight {weight[choices[:, 0] == n_states].sum():.2f}), end of the signal {int(d['signal_end'])}", fontsize=8)
                if ipz == n_pz - 1:
                    ax.set_xlabel("last t of the fit", fontsize=8)
            axes[ipz, 0].set_ylabel(f"pz {pz}:  E1 - E0", fontsize=8)
        axes[0, -1].legend(fontsize=6.5, frameon=False, ncol=3, loc="upper left")
        fig.suptitle(f"{plot_title}: the 2pt gap E1 - E0 of {pt2_set_text[name]} against the fit range\n"
                     "x: last t, color: first t;  left 2 states, right 3 states;  dot size: AIC weight (mean over samples);  error bars: jackknife\n"
                     "y range: 2-98 % of the fits with error < 0.5;  grey band, the same in both columns: the AIC average of the gap over all fits, 2 and 3 states together", fontsize=9.5)
        fig.tight_layout(rect=(0, 0, 1, 0.955))
        save_figure(fig, pdf, "3_pt2", f"pt2_gap_vs_fit_range_{name}")
print("saved", pdf_file, "and the PNGs in", os.path.join(plot_dir, "3_pt2"))